# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Connect to the warehouse
Same Hugging Face + DuckDB connection as Week 4 — reused unchanged so this notebook can
run independently, without needing `w04_baseline_score.ipynb` open in the same session.

In [2]:

%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print('connected')

connected


## Register source tables

In [3]:

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print('tables registered')

tables registered


## Week-4 rule constants (unchanged)
Carried over exactly from the Week-4 baseline so this notebook compares against the
identical rule, not a drifted copy of it. Any change to these values would break the
apples-to-apples comparison in Section 3.

In [6]:

WINDOW_DAYS       = 15
MIN_PREV_IMPR     = 50
IMPR_DROP_THRESH  = 0.20
CLICK_DROP_THRESH = 0.20
POS_SLIP_THRESH   = 1.0
ACTIVITY_DROP_FRAC= 0.25

W_IMPR, W_CLICK, W_POS, W_ACTIVITY = 0.40, 0.25, 0.20, 0.15
assert abs((W_IMPR + W_CLICK + W_POS + W_ACTIVITY) - 1.0) < 1e-9

REASON_CODES = ['IMPR_DROP', 'CLICK_DROP', 'POSITION_SLIP', 'ACTIVITY_DROP']
print('rule constants set')

rule constants set


In [7]:

windowed = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS decision_day FROM {TABLES['fact_daily']}
    ),
    per_item AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            SUM(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_impressions ELSE 0 END)                              AS imp_last,
            SUM(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_impressions ELSE 0 END)                              AS imp_prev,
            SUM(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_clicks ELSE 0 END)                                   AS clk_last,
            SUM(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_clicks ELSE 0 END)                                   AS clk_prev,
            AVG(CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_avg_position END)                                    AS pos_last,
            AVG(CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     THEN f.gsc_avg_position END)                                    AS pos_prev,
            COUNT(DISTINCT CASE WHEN f.report_date >  b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     AND f.gsc_impressions > 0 THEN f.report_date END)               AS active_days_last,
            COUNT(DISTINCT CASE WHEN f.report_date <= b.decision_day - INTERVAL ({WINDOW_DAYS}) DAY
                     AND f.gsc_impressions > 0 THEN f.report_date END)               AS active_days_prev,
            MAX(b.decision_day)                                                      AS decision_day
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.decision_day - INTERVAL ({2*WINDOW_DAYS}) DAY
          AND f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT * FROM per_item
    WHERE imp_prev >= {MIN_PREV_IMPR}
""").df()

d = windowed.copy()
d['pct_impr_change']  = (d['imp_last'] - d['imp_prev']) / d['imp_prev']
d['pct_click_change'] = np.where(d['clk_prev'] > 0, (d['clk_last'] - d['clk_prev']) / d['clk_prev'], np.nan)
d['pos_change']       = d['pos_last'] - d['pos_prev']
d['activity_change']  = (d['active_days_prev'] - d['active_days_last']) / d['active_days_prev'].clip(lower=1)

d['impr_drop_score']     = (-d['pct_impr_change']).clip(lower=0, upper=1)
d['click_drop_score']    = (-d['pct_click_change']).clip(lower=0, upper=1).fillna(0)
d['pos_slip_score']      = (d['pos_change'] / 5.0).clip(lower=0, upper=1)
d['activity_drop_score'] = d['activity_change'].clip(lower=0, upper=1)

def reasons(row):
    fired = []
    if row['pct_impr_change'] <= -IMPR_DROP_THRESH: fired.append('IMPR_DROP')
    if pd.notna(row['pct_click_change']) and row['pct_click_change'] <= -CLICK_DROP_THRESH: fired.append('CLICK_DROP')
    if row['pos_change'] >= POS_SLIP_THRESH: fired.append('POSITION_SLIP')
    if row['activity_change'] >= ACTIVITY_DROP_FRAC: fired.append('ACTIVITY_DROP')
    return fired

d['reason_codes'] = d.apply(reasons, axis=1)
d['n_reasons']     = d['reason_codes'].apply(len)

print(f'{len(d):,} total scored items | {(d["n_reasons"]>0).sum():,} flagged by the rule')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

99,893 total scored items | 78,630 flagged by the rule


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd
import json, os

SEED = 42

# d2 = the full scored population from Week 4 (windowed -> d), not just the flagged 78,630 —
# a fair classifier needs negatives that never crossed the rule's threshold.
d2 = d.copy()

d2['high_confidence_decline'] = (
    (d2['n_reasons'] > 0) & (d2['imp_prev'] >= MIN_PREV_IMPR * 5)
).astype(int)

print(d2['high_confidence_decline'].value_counts(normalize=True).round(4))


high_confidence_decline
0    0.5814
1    0.4186
Name: proportion, dtype: float64




**Lane:** Refresh / Content Opportunity Scoring (continued from Week 4's ranked action engine).

**Target (`y`):** `high_confidence_decline` — 1 if the Week-4 rule flagged the item
(`n_reasons > 0`) **and** its prior-window impressions were comfortably clear of the
reliability floor (`imp_prev >= MIN_PREV_IMPR * 5`). This turns the manual judgment call
from the Week-4 top-20 review ("real volume behind this drop" vs. "near reliability floor")
into a labeled target.

**Method: Logistic Regression.** Reasons this fits the lane:
- It stays decision support, same as the rule — a probability that re-ranks the existing
  queue, not a replacement paradigm.
- Coefficients map one-to-one to the reason-code signals, so the model stays inspectable
  in plain words instead of becoming a black box.
- Week-4's own Section 4 finding was **scale blindness**: percentage-based thresholds
  score a 52-impression blip almost identically to a 3,772-impression collapse. A linear
  model with `log(prior impressions)` added as a feature directly tests whether volume,
  not just percentage change, separates real declines from floor noise — no more
  complexity than the question requires.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(d2, groups=d2['client_hash_id']))

overlap = set(d2.iloc[train_idx]['client_hash_id']) & set(d2.iloc[test_idx]['client_hash_id'])
print('client overlap between train/test:', len(overlap), '(must be 0)')
print('train rows:', len(train_idx), '| test rows:', len(test_idx))

client overlap between train/test: 0 (must be 0)
train rows: 68729 | test rows: 31164




**Grouped by `client_hash_id`, not by row.** Content items from the same client share
site-wide events — a migration, a Core Update, a CMS outage — so a row-level split would
let the model see one item from a client in train and its sibling in test, leaking the
shared event across the split. Week 4's own top-20 review found 45% of the top flagged
items came from a single client — exactly the kind of concentration that makes a
row-level split dishonest here. `GroupShuffleSplit` on `client_hash_id` keeps every
client entirely on one side.

Not time-aware: both Week-4 windows are already fully in the past relative to
`decision_day`, so there's no future boundary to protect — the leakage risk in this lane
is cross-client, not cross-time.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FEATURES = ['pct_impr_change', 'pct_click_change', 'pos_change', 'activity_change', 'log_imp_prev']
d2['log_imp_prev'] = np.log1p(d2['imp_prev'])

train = d2.iloc[train_idx].dropna(subset=FEATURES + ['high_confidence_decline'])
test  = d2.iloc[test_idx].dropna(subset=FEATURES + ['high_confidence_decline']).copy()

X_train, y_train = train[FEATURES], train['high_confidence_decline']
X_test,  y_test  = test[FEATURES],  test['high_confidence_decline']

clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
clf.fit(X_train, y_train)
test['model_score'] = clf.predict_proba(X_test)[:, 1]

# baseline = rule's own flag, same test rows
test['rule_score'] = (test['n_reasons'] > 0).astype(int)

def precision_at_k(df, score_col, k=20):
    return df.sort_values(score_col, ascending=False).head(k)['high_confidence_decline'].mean()

results = pd.DataFrame({
    'method': ['Week-4 rule (baseline)', 'Logistic Regression'],
    'PR-AUC': [
        average_precision_score(y_test, test['rule_score']),
        average_precision_score(y_test, test['model_score']),
    ],
    'precision@20': [
        precision_at_k(test, 'rule_score', 20),
        precision_at_k(test, 'model_score', 20),
    ],
})
print(results.to_string(index=False))

os.makedirs('work/outputs', exist_ok=True)
metrics = {
    "seed": SEED,
    "split": "GroupShuffleSplit on client_hash_id, test_size=0.2",
    "target": "high_confidence_decline (n_reasons>0 AND imp_prev >= MIN_PREV_IMPR*5)",
    "n_train": len(train),
    "n_test": len(test),
    "baseline": {
        "method": "Week-4 rule (n_reasons>0)",
        "pr_auc": float(results.loc[0, 'PR-AUC']),
        "precision_at_20": float(results.loc[0, 'precision@20']),
    },
    "model": {
        "method": "LogisticRegression(class_weight='balanced', random_state=42)",
        "features": FEATURES,
        "pr_auc": float(results.loc[1, 'PR-AUC']),
        "precision_at_20": float(results.loc[1, 'precision@20']),
    },
}
with open('work/outputs/w05_model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('wrote work/outputs/w05_model_metrics.json')

                method   PR-AUC  precision@20
Week-4 rule (baseline) 0.759199          0.60
   Logistic Regression 0.893122          0.35
wrote work/outputs/w05_model_metrics.json


Baseline = the Week-4 rule itself (`n_reasons > 0`), scored on the **same test rows**,
against the **same target**. Metric: PR-AUC and precision@20 — not accuracy, because the
real deliverable is a ranked queue a human works top-down, and accuracy treats every rank
position as equally important when it isn't. precision@20 mirrors the exact top-20 review
from Week 4, so this is an apples-to-apples continuation, not a different exercise.

**Observed result:** PR-AUC rose from 0.759 (rule) to 0.893 (model) — the model separates
the two classes better across the full ranked list. precision@20 fell from 0.60 (rule) to
0.35 (model) — the model's single highest-confidence 20 picks are less concentrated in
genuine high-volume declines than the rule's own top 20. Both numbers are real; the model
is not a strict improvement, it's a different trade-off, and Section 4 digs into why.

Honest framing: the rule has recall = 1 on this target by construction (the target is a
subset of what the rule already flags), so it can't be beaten on recall. The fair test is
whether the model's top-ranked items concentrate more on genuine high-volume declines
than the rule's undifferentiated flag list — and on precision@20, specifically, they
currently don't.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

preds = (test['model_score'] >= 0.5).astype(int)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds, digits=3))

coef_table = pd.Series(clf.coef_[0], index=FEATURES).sort_values()
print("\nCoefficients:\n", coef_table)

perm = permutation_importance(clf, X_test, y_test, n_repeats=20, random_state=SEED)
print("\nPermutation importance:\n", pd.Series(perm.importances_mean, index=FEATURES).sort_values())
fn = test[(y_test == 1) & (preds == 0)].sort_values('model_score')
# Drop identifying columns before display — the pattern (large volume + rising
# impressions) is what matters here, not which client/item it belongs to.
fn[['imp_prev','pct_impr_change','model_score']].head(10).reset_index(drop=True)


[[ 6395  1673]
 [ 2782 11166]]
              precision    recall  f1-score   support

           0      0.697     0.793     0.742      8068
           1      0.870     0.801     0.834     13948

    accuracy                          0.798     22016
   macro avg      0.783     0.797     0.788     22016
weighted avg      0.806     0.798     0.800     22016


Coefficients:
 activity_change    -1.477267
pct_impr_change    -1.448563
pct_click_change   -0.646074
pos_change          0.003069
log_imp_prev        0.901282
dtype: float64

Permutation importance:
 pos_change         -0.000927
activity_change     0.007286
pct_click_change    0.030192
pct_impr_change     0.066402
log_imp_prev        0.186301
dtype: float64


,imp_prev,pct_impr_change,model_score
0,341.0,19.272727,7.768293e-14
1,2492.0,20.004414,1.797209e-13
2,294.0,7.527211,3.952148e-06
3,296.0,6.118243,4.614038e-06
4,705.0,7.059574,4.966859e-05
5,374.0,5.278075,2.240648e-04
6,359.0,2.083565,5.013395e-04
7,1270.0,4.725984,6.868707e-04
8,820.0,2.060976,7.114035e-04
9,315.0,4.834921,1.037424e-03


**Aggregate performance.** On the held-out (client-grouped) test set, the model reaches
0.798 accuracy, with recall 0.801 and precision 0.870 on the positive class
(`high_confidence_decline`). Errors split roughly evenly by direction: 1,673 false
positives and 2,782 false negatives, out of 22,016 test rows.

**PR-AUC vs. precision@20 — two different questions, two different answers.** PR-AUC
rose from 0.759 (rule) to 0.893 (model) — the model separates the two classes better
*overall*. But precision@20 fell from 0.60 (rule) to 0.35 (model) — the model's single
highest-confidence picks are *less* likely to be true high-confidence declines than the
rule's. Both are true at once: the model is a better ranker across the whole queue, but
its most extreme scores cluster on a different, narrower slice of cases than the rule's
top 20 did (see false negatives below). This is a genuine, observed trade-off, not a
clean win — worth flagging to a reviewer rather than smoothing over.

**What the model leans on.** The coefficients show `activity_change` (-1.48) and
`pct_impr_change` (-1.45) as the two strongest signals, with `log_imp_prev` (+0.90) close
behind and `pct_click_change` (-0.65) meaningfully weaker; `pos_change` is effectively
zero (0.003). Permutation importance on held-out data tells a different story about
*practical* reliance: `log_imp_prev` dominates (0.186), then `pct_impr_change` (0.066),
`pct_click_change` (0.030), `activity_change` (0.007), and `pos_change` (~0, even
slightly negative). Read together: the coefficient size for `activity_change` is large,
but the feature doesn't vary enough in the data to matter much in practice, while
`log_imp_prev` — the exact volume signal Week 4's rule structurally lacked — is doing
most of the real separating work. That's the intended fix for scale blindness showing up
directly in the numbers. Average position change carries almost no signal either way,
by either measure.

**False-negative pattern.** The ten lowest-scored false negatives share a distinct
shape: large prior volume (`imp_prev` in the hundreds to low thousands) paired with a
*positive* `pct_impr_change` — impressions actually grew, in some cases by 5–20x. These
items were flagged by the Week-4 rule through a different reason code entirely (a click
or position signal firing independent of impression volume — e.g. CTR compression severe
enough to trigger `CLICK_DROP` even while impressions rose). Because `log_imp_prev` and
`pct_impr_change` dominate the model's decision, a rising-impressions item gets pushed
toward a near-zero score almost automatically, regardless of what the other three signals
say. In plain words: the model has partially overfit to "impressions falling" as its
mental model of decline, and misses the narrower case of a page holding or gaining
visibility while still losing

## Self-check

Before you submit, confirm each line honestly:

-  Every section above is filled — markdown thinking AND the code that backs it
-  The notebook runs top to bottom with no errors (Runtime → Run all)
-  No client names, URLs, or private queries anywhere
-  My claims use careful words: observed, measured, directional, decision-support
-  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.